# Stage 2 / 09 — Evaluate and profile Stage 2 fusion checkpoint

This notebook profiles the checkpoint from notebook 06 and leaves clear hooks for video input. It does not pretend to compute detection mAP unless detection labels/checkpoint outputs are present.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
%cd {REPO_ROOT}
!pip install -q yacs tqdm pyyaml opencv-python-headless tensorboard

In [ ]:
import os, subprocess, json
DRIVE_TRAINING_RUNS = '/content/drive/MyDrive/EcoCAR/training_runs'
CHECKPOINT_TAR = os.path.join(DRIVE_TRAINING_RUNS, 'stage2_rmt_clrkd_basic_fusion.tar')
LOCAL = '/content/stage2_basic_fusion_eval'
assert os.path.exists(CHECKPOINT_TAR), CHECKPOINT_TAR
!rm -rf {LOCAL}
!mkdir -p {LOCAL}
!tar -xf {CHECKPOINT_TAR} -C {LOCAL}
print(os.listdir(LOCAL))
if os.path.exists(os.path.join(LOCAL, 'metrics.json')):
    metrics = json.load(open(os.path.join(LOCAL, 'metrics.json')))
    print(json.dumps(metrics[-1], indent=2))

In [ ]:
import time, torch
x = torch.randn(1, 3, 384, 640, device=device)
with torch.no_grad():
    for _ in range(20):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(100):
        _ = model(x)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    dt = (time.time() - t0) / 100.0
print('latency_ms:', dt * 1000)
print('fps:', 1.0 / dt)
if torch.cuda.is_available():
    print('max_memory_MB:', torch.cuda.max_memory_allocated() / 1024 / 1024)